# Look up instances by video id

Every labelled WLASL-2000 instance (the other splits are subsets of it), keyed by `video_id`: its
gloss, set, signer and source, and its clip at its real length and speed. For browsing a whole
class, see [class_viewer.ipynb](./class_viewer.ipynb). Clips show the labelled frame range, so an
instance that preprocessing removed (see [WLASL_info.md](../../info/WLASL_info.md)) can't be
looked up.

In [1]:
import re
from collections.abc import Iterable

import pandas as pd
from IPython.display import display

from src.preprocess import Instance
from src.video_dataset import load_split_instances
from src.visualise2 import (
    animate_clip_grid,
    animate_frames,
    animation_html,
    load_instance_frames,
    set_thesis_style,
)

set_thesis_style()
instances = load_split_instances("asl2000")


def parse_ids(text: str) -> list[str]:
    """The 5-digit video ids in `text`, in order, e.g. a row of `duplicate_clips.ipynb`'s
    collisions table (`17016, 42541 | 17017, 42542`)."""
    return re.findall(r"\b\d{5}\b", text)


def look_up(video_ids: Iterable[str]) -> list[Instance]:
    """The instance of each video id, raising on any that isn't labelled."""
    ids = list(video_ids)
    missing = [v for v in ids if v not in instances]
    if missing:
        raise KeyError(f"Not labelled in asl2000: {missing}")
    return [instances[v] for v in ids]


def describe(found: list[Instance]) -> pd.DataFrame:
    """One row of metadata per instance."""
    return pd.DataFrame(
        [
            {
                "video_id": i.video_id,
                "gloss": i.label_name,
                "set": i.split,
                "signer": i.signer_id,
                "variation": i.variation_id,
                "source": i.source,
                "frames": i.frame_end - i.frame_start,
                "url": i.url,
            }
            for i in found
        ]
    ).set_index("video_id")


len(instances)

21095

## One video

In [2]:
video_id = "05741"  # change to look up another video
(instance,) = look_up([video_id])
with pd.option_context("display.max_colwidth", None):
    display(describe([instance]))
display(animation_html(*animate_frames(load_instance_frames(instance))))

,gloss,set,signer,variation,source,frames,url
video_id,,,,,,,
05741,before,test,11,0,signingsavvy,28,https://www.signingsavvy.com/signs/mp4/5/5971.mp4


## Several videos in a grid

Give `video_ids` as a list, or paste text containing them into `parse_ids`. The clips play in
step; shorter ones hold their last frame.

In [3]:
video_ids = parse_ids("17016, 42541 | 17017, 42542")  # or a list: ["05741", "41452"]
cols = 4
found = look_up(video_ids)
with pd.option_context("display.max_colwidth", None):
    display(describe(found))
titles = [f"{i.video_id}: {i.label_name}\n{i.split}, signer {i.signer_id}" for i in found]
clips = [load_instance_frames(i) for i in found]
display(animation_html(*animate_clip_grid(clips, cols, titles=titles)))

,gloss,set,signer,variation,source,frames,url
video_id,,,,,,,
17016,doctor,val,60,1,signschool,89,https://signstock.blob.core.windows.net/signschool/videos/db_uploads/SignSchool%20Doctor-zl9aTqYRVXQ.mp4
42541,physician,test,60,0,signschool,89,https://signstock.blob.core.windows.net/signschool/videos/db_uploads/SignSchool%20Doctor-zl9aTqYRVXQ.mp4
17017,doctor,train,21,1,asldeafined,73,https://www.asldeafined.com/media/vocabulary/1467772429.4932.mp4
42542,physician,train,21,0,asldeafined,73,https://www.asldeafined.com/media/vocabulary/1467772429.4932.mp4
